## 1. Initialize the project

Create the working context for the Project, if it has not already been created. The project serves as a placeholder for the code, data, and management of data operations inside the platform.

### Install DigitalHub libraries

If your notebook environment does not already have the DigitalHub libraries installed, run the next cell to install them before continuing. Skip it if the required libraries are already available.

In [30]:
!pip install digitalhub>=0.16 digitalhub-runtime-python digitalhub-runtime-modelserve

In [25]:
import digitalhub as dh
PROJECT_NAME = "llm-lora-adapter"
proj = dh.get_or_create_project(PROJECT_NAME)

Note: Make sure to replace <YOUR_PROJECT_NAME> with the actual name of your project before running the code.

## Prerequisites

Before running the dataset creation and fine-tuning jobs, create a Hugging Face access token and store it as a project secret named `HF_TOKEN`.

1. Open [Hugging Face Settings → Access Tokens](https://huggingface.co/settings/tokens).
2. Generate a new token.
3. Assign the required permissions for model and dataset access.
4. Copy the token value.
5. In the current project, create a secret named `HF_TOKEN` and paste the token as its value.

> The jobs below use `HF_TOKEN` to authenticate against Hugging Face. Do not hardcode the token in the notebook or source files.

In [ ]:
secret0 = proj.new_secret(name="HF_TOKEN", secret_value="put_your_hf_token_here")

## 2. Deploy Training Function


Create the "train" operation in the project. Train function is a python job that performs LLM fine tuning from a predefined HuggingFace model and dataset.

In [28]:
function_train = proj.new_function(
    # image=image,
    name="train-llm", 
    kind="python", 
    python_version="PYTHON3_11", 
    code_src="train.py",
    handler="train_and_log_model",
    requirements="requirements.txt"
)

In [26]:
# function_train = proj.get_function("train-llm")

In [27]:
# image = function_train.refresh().spec.image
# image

Build the function as shown below.

In [29]:
func_b = function_train.run(action='build')

## 3. Perform Training
Training parameters define the model and dataset to be used. Training run refers to HF_TOKEN secret and expects a volume to be created of an appropriate size.

In [ ]:
profile = dh.get_k8s_resource_profiles()
profile

### Select a GPU profile

The `profile` list above contains the resource profiles available through the platform API. Select a GPU profile for the training job; this run uses `1xrtx5000`. Check that the selected profile is available to your project before submitting the job.

In [31]:
train_run = function_train.run(action="job",
                       profile="1xrtx5000",
                       parameters={
                         "model_id": "meta-llama/Llama-3.1-8B",
                         "model_name": "llmpa-tracked",
                         "hf_dataset_name": "team-bay/data-science-qa",
                         "train_data_path": "DataScienceDataset.csv",
                         "dev_data_path": "DataScienceDataset.csv",
                         "num_epochs": 2
                         },
                       secrets=["HF_TOKEN"],
                       envs=[
                          {"name": "HF_HOME", "value": "/app/local_data/huggingface"},
                          {"name": "TRANSFORMERS_CACHE", "value":  "/app/local_data/huggingface"}
                          ],
                       volumes=[{
                          "volume_type": "persistent_volume_claim",
                          "name": "volume-llmpa",
                          "mount_path": "/app/local_data",
                          "spec": { "size": "20Gi" }}]
					)

## 4. Serving the Model (REST API)

Deploy the fine-tuned model using DigitalHub's `vllmserve-text` runtime:

In [ ]:
model = proj.get_model("llmpa-tracked")
model.key

### Base Adapter Model Prerequisite

Before deploying the serving function, log the base adapter model to the platform. Use its platform URL in the `adapters` configuration below.

In [ ]:
base_model = proj.new_model(name="base-model", kind="huggingface", path="hf://meta-llama/Llama-3.1-8B")
base_model.key

'store://llm-lora-adapter/model/huggingface/base-model:1ab298972b2349b691a1da7f38c06e3c'

In [23]:
vllm_func = proj.new_function(
    name="vllm-text-serve",
    kind="vllmserve-text",
    model_name="base-model",
    url=base_model.key,
    adapters = [{
    "name": "adapter",
    "url": model.key
    }],
    
)

In [24]:
serve_run = vllm_func.run(
    action="serve",
    profile="1xrtx5000",
    secrets=["HF_TOKEN"],
    resources={
        "mem": "16Gi"
        },
    wait=True
)

2026-10-14 08:49:52,428 - dhcore./home/khurshid/python3.12/lib/python3.12/site-packages/digitalhub_runtime_modelserve/entities/run/modelserve_run/entity.py - INFO - Waiting for run b046294bdf624d57bee38d99b9ea9461 to deploy service.
2026-10-14 08:49:52,544 - dhcore./home/khurshid/python3.12/lib/python3.12/site-packages/digitalhub_runtime_modelserve/entities/run/modelserve_run/entity.py - INFO - Run b046294bdf624d57bee38d99b9ea9461 serving failed.


### Inference Request Example

Once deployed, send POST requests to the serving endpoint:


In [ ]:
service = serve_run.refresh().status.service
model_name = serve_run.status.openai.get("model", "llmpa-tracked") if serve_run.status.openai else "llmpa-tracked"

payload = {
    "model": model_name,
    "prompt": "Explain gradient descent in machine learning.",
    "max_tokens": 256,
    "temperature": 0.7,
}

endpoint_url = f"http://{service['url']}/v1/completions" if service and "url" in service else None
response = serve_run.invoke(json=payload, url=endpoint_url).json()

print("Response from served model:")
print(response)